# Project 2: Section-Scoped RAG System for SEC 10-K (2024)

**Domain:** AI Strategy / AI Investment  
**SEC Items Used:** Item 1 (Business), Item 1A (Risk Factors), Item 7 (MD&A)  
**Embedding Model:** `sentence-transformers/all-MiniLM-L6-v2` (local, free)  
**Generation Model:** Github API — swappable with OpenAI/Gemini

**Vector Store:** FAISS  

---
**Why this scope?**  
Focusing on AI-related language in Items 1, 1A, and 7 gives a dense, semantically coherent corpus. Retrieval works best when chunks share a common vocabulary a narrow domain prevents noise from unrelated financial disclosures polluting results.

## Step 0 — Install Dependencies

In [ ]:
# Run once at the top of your Colab session
!pip install -q sentence-transformers faiss-cpu anthropic langchain langchain-community pandas tqdm beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


## Step 1 — Connect Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

# ── path ──────────────────────────────────
INPUT_DIR  = Path("/content/drive/MyDrive/Gen/SEC-10K-2024-TXT")
OUTPUT_DIR = Path("/content/drive/MyDrive/SEC-10K-2024-ITEMS")  # extracted item files
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

html_files = sorted(INPUT_DIR.glob("*.txt"))
print(f"Found {len(html_files)} 10-K text files")
html_files[:5]

Found 0 10-K text files


[]

In [ ]:
print(f"Listing contents of {INPUT_DIR}:")
!ls -F {INPUT_DIR}

Streaming output truncated to the last 5000 lines.
20240308_10-K_edgar_data_1325702_0001193125-24-063674.txt
20240308_10-K_edgar_data_1327811_0001327811-24-000044.txt
20240308_10-K_edgar_data_1331465_0001331465-24-000046.txt
20240308_10-K_edgar_data_1367644_0001367644-24-000033.txt
20240308_10-K_edgar_data_1369290_0000950170-24-028326.txt
20240308_10-K_edgar_data_1370450_0001558370-24-002758.txt
20240308_10-K_edgar_data_1371489_0001558370-24-002831.txt
20240308_10-K_edgar_data_1378950_0001378950-24-000034.txt
20240308_10-K_edgar_data_1394638_0001731122-24-000378.txt
20240308_10-K_edgar_data_1400810_0000950170-24-028620.txt
20240308_10-K_edgar_data_1405513_0001410578-24-000158.txt
20240308_10-K_edgar_data_1412665_0001412665-24-000039.txt
20240308_10-K_edgar_data_1425450_0001425450-24-000013.txt
20240308_10-K_edgar_data_1436425_0001436425-24-000009.txt
20240308_10-K_edgar_data_1452857_0001452857-24-000008.txt
20240308_10-K_edgar_data_1454938_0001454938-24-000024.txt
20240308_10-K_edgar_d

## Step 2 — Extract SEC Item Sections from Raw HTML

We pull Items 1, 1A, and 7 — the sections most likely to contain AI strategy language.

In [ ]:
import re
from tqdm import tqdm

# ── Item boundary map ─────────────────────────────────────────────────────────
ITEM_BOUNDARIES = {
    "1":  ["1A", "1B", "2"],
    "1A": ["1B", "2", "3"],
    "3":  ["4", "5"],
    "7":  ["7A", "8", "9"],
    "7A": ["8", "9"],
    "11": ["12", "13", "14"],
}

# Items to extract for the AI-strategy domain
SELECTED_ITEMS = ["1", "1A", "7"]


def normalize_text_for_matching(text: str) -> str:
    text = re.sub(r'\xa0', ' ', text)
    text = re.sub(r'&nbsp;', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text


def read_html_file(file_path) -> str:
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def extract_item_section(html_text: str, item_label: str, next_item_labels: list) -> str | None:
    """Return the text slice for one SEC item, stopping before the next item."""
    text = normalize_text_for_matching(html_text)
    start_pat = re.compile(
        rf'item\s*{re.escape(item_label)}[\.\:\-\s]', re.IGNORECASE
    )
    start_match = start_pat.search(text)
    if not start_match:
        return None

    start_idx = start_match.start()
    end_idx   = len(text)

    for next_label in next_item_labels:
        next_pat   = re.compile(
            rf'item\s*{re.escape(next_label)}[\.\:\-\s]', re.IGNORECASE
        )
        next_match = next_pat.search(text, pos=start_match.end())
        if next_match:
            end_idx = min(end_idx, next_match.start())

    return text[start_idx:end_idx].strip()


def save_extracted_items(file_path, selected_items, output_dir) -> list:
    html_text  = read_html_file(file_path)
    base_name  = file_path.stem
    saved_files = []

    for item in selected_items:
        next_items = ITEM_BOUNDARIES.get(item, [])
        extracted  = extract_item_section(html_text, item, next_items)
        if extracted and len(extracted) > 500:   # skip tiny / false matches
            out_file = output_dir / f"{base_name}_item_{item}.html"
            with open(out_file, "w", encoding="utf-8") as f:
                f.write(extracted)
            saved_files.append(str(out_file))

    return saved_files


# Run over all filings
all_saved = []
for file_path in tqdm(html_files, desc="Extracting sections"): # Added tqdm for progress bar
    saved = save_extracted_items(file_path, SELECTED_ITEMS, OUTPUT_DIR)
    all_saved.extend(saved)

print(f"Saved {len(all_saved)} extracted item files.")
all_saved[:10]

Extracting sections: 100%|██████████| 7754/7754 [29:57<00:00,  4.31it/s]

Saved 1400 extracted item files.


['/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240102_10-K_edgar_data_90168_0000090168-23-000083_item_1.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240102_10-K_edgar_data_90168_0000090168-23-000083_item_1A.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240102_10-K_edgar_data_90168_0000090168-23-000083_item_7.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240105_10-K_edgar_data_315374_0001558370-24-000115_item_1.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240105_10-K_edgar_data_315374_0001558370-24-000115_item_1A.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240105_10-K_edgar_data_315374_0001558370-24-000115_item_7.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240111_10-K-A_edgar_data_1858007_0001213900-24-003029_item_1A.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240116_10-K_edgar_data_1555214_0001493152-24-002416_item_1.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240116_10-K_edgar_data_1555214_0001493152-24-002416_item_1A.ht

## Step 3 — Clean & Chunk the Extracted Text

In [ ]:
from pathlib import Path

# Ensure OUTPUT_DIR is defined, in case it's run out of order
OUTPUT_DIR = Path("/content/drive/MyDrive/SEC-10K-2024-ITEMS")

# Populate all_saved with the .html files in OUTPUT_DIR as strings
all_saved = [str(p) for p in sorted(OUTPUT_DIR.glob("*.html"))]
print(f"Found {len(all_saved)} previously saved extracted item files in OUTPUT_DIR.")
all_saved[:5]

Found 1400 previously saved extracted item files in OUTPUT_DIR.


['/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240102_10-K_edgar_data_90168_0000090168-23-000083_item_1.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240102_10-K_edgar_data_90168_0000090168-23-000083_item_1A.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240102_10-K_edgar_data_90168_0000090168-23-000083_item_7.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240105_10-K_edgar_data_315374_0001558370-24-000115_item_1.html',
 '/content/drive/MyDrive/SEC-10K-2024-ITEMS/20240105_10-K_edgar_data_315374_0001558370-24-000115_item_1A.html']

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

CHUNK_SIZE    = 400   # tokens (approx words)
CHUNK_OVERLAP = 50


def html_to_plain_text(html_snippet: str) -> str:
    """Strip HTML tags and return clean plain text."""
    soup = BeautifulSoup(html_snippet, "html.parser")
    return soup.get_text(separator=" ", strip=True)


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    """Split text into overlapping word-level chunks."""
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        end   = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks


records = []

for filepath in all_saved:
    p         = Path(filepath)
    parts     = p.stem.split("_item_")           # e.g. ["AAPL_10K", "1A"]
    base_name = parts[0]
    section   = parts[1] if len(parts) > 1 else "unknown"

    raw_html  = p.read_text(encoding="utf-8", errors="ignore")
    plain     = html_to_plain_text(raw_html)
    chunks    = chunk_text(plain)

    for i, chunk in enumerate(chunks):
        records.append({
            "chunk_id":     f"{base_name}_item{section}_chunk{i:04d}",
            "company_name": base_name,
            "section_id":   f"Item {section}",
            "token_count":  len(chunk.split()),
            "text":         chunk,
        })

df = pd.DataFrame(records)
print(f"Total chunks: {len(df)}")
df.head(3)

Total chunks: 39271


,chunk_id,company_name,section_id,token_count,text
0,20240102_10-K_edgar_data_90168_0000090168-23-0...,20240102_10-K_edgar_data_90168_0000090168-23-0...,Item 1,400,Item 1. Business A. The Company SIFCO Industri...
1,20240102_10-K_edgar_data_90168_0000090168-23-0...,20240102_10-K_edgar_data_90168_0000090168-23-0...,Item 1,400,in commercial revenues and 52.6 in military re...
2,20240102_10-K_edgar_data_90168_0000090168-23-0...,20240102_10-K_edgar_data_90168_0000090168-23-0...,Item 1,400,its ability to pass through raw material costs...


## Step 4 — Inspect the Corpus Before Embedding

In [ ]:
# Save the processed DataFrame to a CSV file for persistence
output_csv_path = OUTPUT_DIR / "processed_chunks.csv"
df.to_csv(output_csv_path, index=False)
print(f"Processed chunks saved to {output_csv_path}")

Processed chunks saved to /content/drive/MyDrive/SEC-10K-2024-ITEMS/processed_chunks.csv


In [ ]:
print(" Corpus Summary ")
print(f"Total chunks    : {len(df)}")
print(f"Unique companies: {df['company_name'].nunique()}")
print(f"Sections        : {df['section_id'].unique().tolist()}")
print(f"Avg token count : {df['token_count'].mean():.0f}")
print(f"Min/Max tokens  : {df['token_count'].min()} / {df['token_count'].max()}")

print(" Section Distribution")
print(df['section_id'].value_counts())

print("Sample Chunks")
for _, row in df.sample(3, random_state=42).iterrows():
    print(f"\n[{row['chunk_id']}]")
    print(row['text'][:300], "...")

 Corpus Summary 
Total chunks    : 39271
Unique companies: 725
Sections        : ['Item 1', 'Item 1A', 'Item 7']
Avg token count : 393
Min/Max tokens  : 51 / 400
 Section Distribution
section_id
Item 1A    18922
Item 7     11724
Item 1      8625
Name: count, dtype: int64
Sample Chunks

[20240415_10-K_edgar_data_1621672_0001437749-24-011939_item7_chunk0059]
Currently, we have one patent application pending, and 187 registered trademarks, along with certain licenses from game publishers to utilize their proprietary games. For our pending patent application, we cannot assure you that we will be granted patents pursuant to our pending applications as well ...

[20240226_10-K_edgar_data_898174_0000898174-24-000048_item1A_chunk0023]
assets of such subsidiary before we, as shareholder, would be entitled to any payment. Our subsidiaries would have to pay their direct creditors in full before our creditors, including holders of common stock, preferred stock or debt securities of RGA, could rece

## Step 5 — Build the Embedding Layer

**Model chosen:** `all-MiniLM-L6-v2`  
- 384-dimensional vectors — compact and fast on Colab CPU/GPU  
- Strong semantic similarity performance for English prose  
- Completely free, no API key required  
- Ideal compute trade-off for a prototype with < 10k chunks

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL}")
embedder = SentenceTransformer(EMBEDDING_MODEL)

texts = df["text"].tolist()

print(f"Encoding {len(texts)} chunks...")
embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # cosine similarity via dot product
)

embeddings = np.array(embeddings, dtype="float32")
print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"Embedding dimension   : {embeddings.shape[1]}")

# Save embeddings to disk for persistence
embeddings_path = OUTPUT_DIR / "embeddings.npy"
np.save(embeddings_path, embeddings)
print(f"Embeddings saved to {embeddings_path}")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
# Load embeddings
OUTPUT_DIR = Path("/content/drive/MyDrive/SEC-10K-2024-ITEMS")
embeddings_path = OUTPUT_DIR / "embeddings.npy"
embeddings = np.load(embeddings_path)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"Embedding dimension   : {embeddings.shape[1]}")


Embedding matrix shape: (39271, 384)
Embedding dimension   : 384


## Step 6 — Build the Vector Store (FAISS)

In [ ]:
import faiss
import numpy as np
from pathlib import Path

# Ensure embeddings are available (either from memory or loaded from disk)
if 'embeddings' not in locals(): # Check if 'embeddings' variable is defined
    embeddings_path = OUTPUT_DIR / "embeddings.npy"
    if embeddings_path.exists():
        embeddings = np.load(embeddings_path)
        print(f"Embeddings loaded from {embeddings_path}")
    else:
        raise FileNotFoundError(
            f"Embeddings file not found at {embeddings_path}. "
            "Please ensure the embedding generation cell (Step 5) is run first to create and save them."
        )

DIM   = embeddings.shape[1]   # 384
TOP_K = 5   # retrieve 5 chunks per query
# Rationale: 5 chunks ≈ 2,000 words of context — enough for an LLM
# to synthesize an answer without overflowing the prompt window.

# Inner Product index (works as cosine similarity because embeddings are L2-normalized)
index = faiss.IndexFlatIP(DIM)
index.add(embeddings)

print(f"FAISS index type  : IndexFlatIP")
print(f"Vectors indexed   : {index.ntotal}")
print(f"top_k             : {TOP_K}")

Embeddings loaded from /content/drive/MyDrive/SEC-10K-2024-ITEMS/embeddings.npy
FAISS index type  : IndexFlatIP
Vectors indexed   : 39271
top_k             : 5


## Step 7 — Retrieval Function

In [ ]:
def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    """Embed query, search FAISS, return top-k chunk dicts with scores."""
    q_vec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = df.iloc[idx]
        results.append({
            "chunk_id":     row["chunk_id"],
            "company_name": row["company_name"],
            "section_id":   row["section_id"],
            "score":        float(score),
            "text":         row["text"],
        })
    return results


# Quick sanity test
test_results = retrieve("AI investment strategy")
for r in test_results:
    print(f"[{r['score']:.3f}] {r['chunk_id']}")
    print(r['text'][:200], "\n")

[0.532] 20240214_10-K_edgar_data_1392972_0001392972-24-000025_item7_chunk0002
as a result, we grew both subscription revenue and total revenue by double digits in 2023. We believe in the near term that new customers will emphasize smaller scope initial purchases and fast return 

[0.497] 20240229_10-K_edgar_data_1398659_0001398659-24-000031_item7_chunk0031
has occurred in the past), could adversely affect our business, growth strategy and results of operations. Developments in the industries we serve, which are increasingly rapid, have shifted and may c 

[0.495] 20240628_10-K_edgar_data_1922947_0001213900-24-056788_item1A_chunk0004
Investment Adviser employs an active investment strategy with a bottom-up approach to portfolio construction. The investment strategy is predicated on an intensive underwriting and due diligence proce 

[0.479] 20240214_10-K_edgar_data_1058290_0001058290-24-000017_item1A_chunk0026
as-a-service solutions, among others. If we do not sufficiently invest in new

## Step 8 — Generation Function



In [ ]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-TinyBERT-L-2" # A lightweight but effective cross-encoder
print(f"Loading reranker model: {RERANKER_MODEL_NAME}")
cross_encoder = CrossEncoder(RERANKER_MODEL_NAME)

def reranker(chunks: list[dict], question: str) -> list[dict]:
    """Rerank retrieved chunks based on their relevance to the question using a cross-encoder."""
    if not chunks:
        return []

    # Prepare sentence pairs for the cross-encoder
    sentence_pairs = [[question, chunk['text']] for chunk in chunks]

    # Predict relevance scores
    relevance_scores = cross_encoder.predict(sentence_pairs)

    # Update scores in chunks and sort
    for i, score in enumerate(relevance_scores):
        chunks[i]['score'] = float(score) # Overwrite the FAISS score with the reranked score

    # Sort chunks by the new score in descending order
    reranked_chunks = sorted(chunks, key=lambda x: x['score'], reverse=True)

    return reranked_chunks

print("Reranker loaded and ready.")

Loading reranker model: cross-encoder/ms-marco-TinyBERT-L-2


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-TinyBERT-L-2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded and ready.


To use the Github's API, you'll need an API key. Add the key to the Colab secrets manager . Give it the name `GITHUB_TOKEN`. Then, the following code will configure the API.

Model chosen: `gpt-4o-mini` as it offers the ideal balance of fast inference, cost-efficiency, and strict instruction-following.

In [ ]:
!pip install langchain langchain_core langchain_community faiss-cpu openai>=1.56.2 langchain_openai sentence-transformers transformers accelerate

In [ ]:
import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')


In [ ]:
import os
from openai import OpenAI

# Initialize the OpenAI client using GitHub Models endpoint
client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.environ.get("GITHUB_TOKEN")
)

SYSTEM_PROMPT = """You are a financial analyst assistant specializing in SEC 10-K reports.
Your primary goal is to provide accurate, concise, and direct answers based *only* on the financial document excerpts provided.

Here are the rules you *must* follow:
1.  **Strictly use provided excerpts:** Only use information found in the 'RETRIEVED CONTEXT' section. Do not use any outside knowledge.
2.  **Cite sources:** For each piece of information, indicate the source chunk ID (e.g., [Chunk 1], [Chunk 2]) at the end of the sentence or paragraph where the information was extracted.
3.  **Be concise:** Provide direct answers without unnecessary conversational filler.
4.  **Handle missing information:** If the answer cannot be found in *any* of the provided excerpts, explicitly state: "I cannot find a direct answer to this question in the provided SEC 10-K excerpts." Do *not* make up information or speculate.
5.  **Focus on the question:** Ensure your answer directly addresses the user's question.
"""

def generate_answer(question: str, chunks: list[dict]) -> str:
    """Assemble context from retrieved chunks and ask the LLM."""
    context_parts = []
    for i, c in enumerate(chunks, 1):
        context_parts.append(
            f"[Chunk {i} | {c['company_name']} | {c['section_id']} | score={c['score']:.3f}]\n"
            f"{c['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    user_message = f"""Use only the following SEC 10-K excerpts to answer the question.

RETRIEVED CONTEXT
{context}
 END CONTEXT

Question: {question}

Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ],
        temperature=0.0,
        max_tokens=600
    )
    return response.choices[0].message.content

def rag_query(question: str, top_k: int = TOP_K) -> dict:
    """Full RAG pipeline: retrieve → assemble → generate."""
    chunks = retrieve(question, top_k)
    reranked_chunks = reranker(chunks, question)
    answer = generate_answer(question, reranked_chunks)
    return {"question": question, "chunks": reranked_chunks, "answer": answer}


print("RAG pipeline ready with GitHub Models API.")

RAG pipeline ready with GitHub Models API.


## Step 9 — Ask Structured Domain Questions

We evaluate the RAG pipeline on a set of 10 domain-specific questions
regarding AI strategy and investment.

Process:
1. Define a list of 10 questions.
    - The evaluation questions are broad, corpus-wide inquiries rather than focusing on specific companies. This was necessary because our current retrieval pipeline does not explicitly route company metadata in the prompts, making generic questions much more effective for evaluating the system's overall performance.
2. Loop through each question and pass it to the `rag_query` function.
3. Retrieve the top_k most relevant chunks (top_k=5).
   Justification for top_k=5: Retrieving 5 chunks (approx. 2000 words/tokens total)
   provides a rich enough context for the LLM to synthesize a comprehensive, factual answer
   without overflowing the context window or diluting the relevance with excess noise.
4. Store the question, generated answer, raw chunk text (as a list), scores, and metadata
   in a structured list.
5. Convert and save the results into a pandas DataFrame for easy inspection and downstream analysis.
6. Handling Retrieval Edge Cases:
   - *Too many chunks:* By hard-coding `top_k=5`, we cap the context limit, ensuring the LLM is never overwhelmed and reducing token costs.
   - *Too few or irrelevant chunks:* Since the FAISS index contains 39271 vectors, a search will always return 5 chunks. However, if a query is completely irrelevant, these chunks will have low similarity scores. We handle this via the LLM's System Prompt ("If the answer cannot be found in the excerpts, say so explicitly"), which prevents hallucinations when retrieval quality is poor.

In [ ]:
import pandas as pd
import time

ai_strategy_questions = [
    "How is AI described as a strategic investment area in the financial disclosures?",
    "Are AI initiatives framed primarily for operational efficiency, product innovation, or both?",
    "What specific AI-related expenditures or R&D activities are quantified or discussed?",
    "What specific risks or uncertainties are identified regarding AI adoption and implementation?",
    "What regulatory, compliance, or data privacy concerns are mentioned regarding the use of AI?",
    "Does the company SIFCO Industries describe AI as a strategic investment area?",
    "How is AI viewed in terms of competitive advantage or competitive threat?",
    "What talent acquisition or retention challenges are discussed specifically related to AI expertise?",
    "What references exist regarding ethical considerations or 'responsible AI' frameworks?",
    "How is generative AI being utilized to enhance customer experience or service offerings?"
]

results_data = []

print("Processing questions through the RAG pipeline...")
for q in ai_strategy_questions:
    print(f"Asking: {q}")
    success = False
    retries = 3
    while not success and retries > 0:
        try:
            result = rag_query(q, top_k=TOP_K) #TOP_K = 5 as defined earlier

            chunks = result['chunks']

            results_data.append({
                "Question": q,
                "Generated_Answer": result['answer'],
                "Retrieved_Chunks": [c['text'] for c in chunks],  #Store chunks as a list of strings
                "Chunk_Scores": [c['score'] for c in chunks],
                "Chunk_Metadata": [{"company": c['company_name'], "section": c['section_id'], "chunk_id": c['chunk_id']} for c in chunks]
            })
            success = True
        except Exception as e:
            print(f"Error generating answer: {e}")
            print("Quota exceeded or error encountered. Waiting 30 seconds before retrying...") # Add this to prevent quota exceeded error that tends to occur with free tier of API keys
            time.sleep(30)
            retries -= 1

    # Wait 15 seconds between successful requests to respect free tier API rate limits
    time.sleep(15)

df_qa_results = pd.DataFrame(results_data)

print("\n--- RAG Evaluation DataFrame ---")
display(df_qa_results.head())


Processing questions through the RAG pipeline...
Asking: How is AI described as a strategic investment area in the financial disclosures?
Asking: Are AI initiatives framed primarily for operational efficiency, product innovation, or both?
Asking: What specific AI-related expenditures or R&D activities are quantified or discussed?
Asking: What specific risks or uncertainties are identified regarding AI adoption and implementation?
Asking: What regulatory, compliance, or data privacy concerns are mentioned regarding the use of AI?
Asking: Does the company SIFCO Industries describe AI as a strategic investment area?
Asking: How is AI viewed in terms of competitive advantage or competitive threat?
Asking: What talent acquisition or retention challenges are discussed specifically related to AI expertise?
Asking: What references exist regarding ethical considerations or 'responsible AI' frameworks?
Asking: How is generative AI being utilized to enhance customer experience or service offering

,Question,Generated_Answer,Retrieved_Chunks,Chunk_Scores,Chunk_Metadata
0,How is AI described as a strategic investment ...,I cannot find a direct answer to this question...,[to mitigate ethical and legal issues presente...,"[0.09171578288078308, 0.05748751759529114, 0.0...",[{'company': '20240222_10-K_edgar_data_743238_...
1,Are AI initiatives framed primarily for operat...,AI initiatives are framed primarily for both o...,[could have a material adverse effect on our b...,"[0.5680254697799683, 0.3927267789840698, 0.283...",[{'company': '20240229_10-K_edgar_data_1398659...
2,What specific AI-related expenditures or R&D a...,"In fiscal 2023 and 2022, the company incurred ...","[development of AI products and technologies, ...","[0.044030219316482544, 0.021750129759311676, 0...",[{'company': '20241029_10-K_edgar_data_1013237...
3,What specific risks or uncertainties are ident...,The specific risks or uncertainties identified...,[could have a material adverse effect on our b...,"[0.8087233901023865, 0.7320768237113953, 0.282...",[{'company': '20240229_10-K_edgar_data_1398659...
4,"What regulatory, compliance, or data privacy c...","The regulatory, compliance, and data privacy c...",[personnel use generative AI technologies to p...,"[0.7412207126617432, 0.7412207126617432, 0.517...",[{'company': '20240318_10-K-A_edgar_data_15681...


In [ ]:
# Save the generated responses to a CSV file for persistence
output_csv_path = OUTPUT_DIR / "domain_qa.csv"
df_qa_results.to_csv(output_csv_path, index=False)
print(f"Responses saved to {output_csv_path}")

Responses saved to /content/drive/MyDrive/SEC-10K-2024-ITEMS/domain_qa.csv


## Step 10 — Advanced Evaluation

RAGAS (Retrieval Augmented Generation Assessment) Evaluation

This step evaluates the quality of the RAG pipeline using two key metrics:
1. Faithfulness: Measures the factual consistency of the generated answer against the retrieved context.
   It checks if all claims made in the answer can be directly inferred from the retrieved chunks (penalizing hallucinations).
2. Answer Relevancy: Measures how directly and appropriately the generated answer addresses the question.
   It penalizes off-topic, incomplete, or redundant answers.

Note: We omit metrics like context_precision and context_recall here because they require a manually annotated 'ground_truth' reference.

In [ ]:
!pip install -q ragas langchain langchain-google-genai datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 17.1 MB/s eta 0:00:00


In [ ]:
import os
import time
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

github_token = os.environ.get("GITHUB_TOKEN")

# Setup the LLM that will perform evluation
ragas_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0, # Since RAGAS uses LLM-as-a-judge to calculate scores, we set temp to 0 which makes LLM's responses deterministic and reproducible
    api_key=github_token,
    base_url="https://models.inference.ai.azure.com"
)

#set up embeddings llm that will calculate semantic similarity (for AnswerRelevancy metric)
ragas_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=github_token,
    base_url="https://models.inference.ai.azure.com"
)

print("Running Ragas evaluation sequentially to avoid timeouts...")

all_results = []
total_questions = len(df_qa_results)

# Running RAGAS to evaluate all questions at once was resulting in time out errors so we implemented sequential processing instead
for i in range(total_questions):
    print(f"Evaluating question {i+1} of {total_questions}...")

    # Create a single row dataset
    single_row_data = {
        "question": [df_qa_results["Question"].iloc[i]],
        "answer": [df_qa_results["Generated_Answer"].iloc[i]],
        "contexts": [df_qa_results["Retrieved_Chunks"].iloc[i]]
    }
    single_dataset = Dataset.from_dict(single_row_data)

    # Evaluate just this row
    result = evaluate(
        single_dataset,
        metrics=[
            Faithfulness(),
            AnswerRelevancy(),
        ],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False
    )

    all_results.append(result.to_pandas())

    # Wait 15s to respect rate limits
    if i < total_questions - 1:
        time.sleep(15)

print("\n--- Ragas Evaluation Results ---")
df_ragas_results = pd.concat(all_results, ignore_index=True)
display(df_ragas_results.head())

# Calculate overall scores
print("\nOverall Average Metrics:")
print(f"Faithfulness: {df_ragas_results['faithfulness'].mean():.4f}")
print(f"Answer Relevancy: {df_ragas_results['answer_relevancy'].mean():.4f}")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_6675/3148697344.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy
/tmp/ipykernel_6675/3148697344.py:6: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
 

Running Ragas evaluation sequentially to avoid timeouts...
Evaluating question 1 of 10...


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating question 2 of 10...


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

...

Evaluating question 10 of 10...


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]


--- Ragas Evaluation Results ---


,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,How is AI described as a strategic investment ...,[to mitigate ethical and legal issues presente...,I cannot find a direct answer to this question...,1.0,0.000000
1,Are AI initiatives framed primarily for operat...,[could have a material adverse effect on our b...,AI initiatives are framed primarily for both o...,0.5,0.687629
2,What specific AI-related expenditures or R&D a...,"[development of AI products and technologies, ...","In fiscal 2023 and 2022, the company incurred ...",1.0,0.512423
3,What specific risks or uncertainties are ident...,[could have a material adverse effect on our b...,The specific risks or uncertainties identified...,1.0,0.979110
4,"What regulatory, compliance, or data privacy c...",[personnel use generative AI technologies to p...,"The regulatory, compliance, and data privacy c...",1.0,0.970914



Overall Average Metrics:
Faithfulness: 0.8500
Answer Relevancy: 0.4808


In [ ]:
df_ragas_results

,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,How is AI described as a strategic investment ...,[to mitigate ethical and legal issues presente...,I cannot find a direct answer to this question...,1.0,0.000000
1,Are AI initiatives framed primarily for operat...,[could have a material adverse effect on our b...,AI initiatives are framed primarily for both o...,0.5,0.687629
2,What specific AI-related expenditures or R&D a...,"[development of AI products and technologies, ...","In fiscal 2023 and 2022, the company incurred ...",1.0,0.512423
3,What specific risks or uncertainties are ident...,[could have a material adverse effect on our b...,The specific risks or uncertainties identified...,1.0,0.979110
4,"What regulatory, compliance, or data privacy c...",[personnel use generative AI technologies to p...,"The regulatory, compliance, and data privacy c...",1.0,0.970914
5,Does the company SIFCO Industries describe AI ...,[A E expertise; (ii) focus on quality and cust...,I cannot find a direct answer to this question...,0.0,0.000000
6,How is AI viewed in terms of competitive advan...,[significant risks involved in using AI and no...,AI is viewed as both a potential competitive a...,1.0,0.000000
7,What talent acquisition or retention challenge...,[best talent in the industry. Our commitment t...,I cannot find a direct answer to this question...,1.0,0.000000
8,What references exist regarding ethical consid...,[significant risks involved in using AI and no...,The excerpts reference ethical considerations ...,1.0,0.714151
9,How is generative AI being utilized to enhance...,[the generative AI surprises pop up across bus...,Generative AI is being utilized to enhance cus...,1.0,0.943597


In [ ]:
# Save RAGAS results to a CSV file for persistence
output_csv_path = OUTPUT_DIR / "df_ragas_results.csv"
df_ragas_results.to_csv(output_csv_path, index=False)
print(f"RAGAS results saved to {output_csv_path}")

RAGAS results saved to /content/drive/MyDrive/SEC-10K-2024-ITEMS/df_ragas_results.csv


The RAG pipeline demonstrates strong Faithfulness, with an average score of ~0.85. For the majority of queries, it scored a perfect 1.0, indicating that the generated answers are highly grounded in the retrieved SEC 10-K contexts with minimal hallucination. The few drops in faithfulness occurred when the system correctly stated it could not find an answer, or occasionally synthesized across chunks.

However, Answer Relevancy showed highly varied performance (averaging around 0.48). On several questions, the system produced highly relevant answers (scoring ~0.70 to 0.97). On others, the system scored 0.0, primarily because it explicitly stated: *"I cannot find a direct answer to this question in the provided SEC 10-K excerpts."* While this behavior perfectly follows the strict system instructions to prevent hallucinations, it penalizes the Answer Relevancy metric. This suggests that while our generation step is very safe and factual, our **retrieval step** (using a simple FAISS IP search with top_k=5) occasionally fails to find the necessary context for more specific or nuanced queries. Future improvements should focus on advanced retrieval techniques like dense/sparse hybrid search or query expansion.


## Step 11 — Manual Evaluation, Grounding Check, and Failure Analysis

This section directly addresses the required Project 2 evaluation criteria. Although the notebook also includes a RAGAS-style quantitative evaluation, the assignment requires a simple manual evaluation of retrieval relevance, grounding, and failure cases. For this review, we inspected the retrieved chunks, chunk scores, metadata, and generated answers for representative AI-domain questions.

### 15.1.1 Retrieval Relevance

The table below summarizes whether the retrieved chunks were relevant and whether they captured the right domain evidence for the question.

| Test Question | Retrieval Relevance | Did retrieval capture the right domain evidence? | Notes |
|---|---|---|---|
| How is AI described as a strategic investment area in the financial disclosures? | Weak / Partial | No direct evidence captured | The model returned an answer saying it could not find a direct answer. Similarity scores were low, so the retrieved chunks were not strong evidence for strategic AI investment. |
| Are AI initiatives framed primarily for operational efficiency, product innovation, or both? | Moderate | Mostly yes | The retrieved chunks supported both operational efficiency and product innovation, but the evidence was spread across different companies and sections. |
| What specific AI-related expenditures or R&D activities are quantified or discussed? | Weak / Partial | Partially | The answer cited R&D expense figures, but similarity scores were low, so this should be treated cautiously unless the retrieved excerpt directly links those R&D expenses to AI-related work. |
| What specific risks or uncertainties are identified regarding AI adoption and implementation? | Strong | Yes | Retrieved chunks were highly relevant and came from risk-oriented language, making this one of the strongest retrieval cases. |
| What regulatory, compliance, or data privacy concerns are mentioned regarding the use of AI? | Strong | Yes | Retrieval captured strong evidence related to AI regulation, privacy, security, compliance costs, and liability exposure. |
| Does the company SIFCO Industries describe AI as a strategic investment area? | Failure case | No | This company-specific query exposed a limitation of the current system. The pipeline does not deeply integrate company metadata into retrieval, so company-specific answers are less reliable. |
| What talent acquisition or retention challenges are discussed specifically related to AI expertise? | Weak | No direct evidence captured | Retrieved similarity scores were extremely low, and the answer correctly stated that no direct evidence was found. |
| How is generative AI being utilized to enhance customer experience or service offerings? | Strong | Yes | Retrieved chunks were highly relevant and supported the answer about customer engagement, contact centers, automation, and service improvement. |

### 15.1.2 Grounding Check

Grounding was evaluated by checking whether the generated answer was supported by the retrieved SEC 10 K excerpts and whether it introduced unsupported claims. Overall, the generation step was conservative and generally avoided hallucination because the system prompt instructed the model to rely only on retrieved context and to explicitly say when the answer could not be found.

| Grounding Question | Assessment |
|---|---|
| Did the answer rely only on retrieved evidence? | Mostly yes. The generated answers generally stayed within the retrieved context, especially for risk, compliance, privacy, and generative AI use-case questions. |
| Did the answer introduce unsupported claims? | Minimal hallucination risk was observed. The system often refused to answer when the retrieved context was weak, which protected grounding but reduced answer usefulness. |
| What did the quantitative evaluation show? | RAGAS faithfulness was strong at approximately 0.85 on average, suggesting most answers were grounded in the retrieved contexts. |
| What was the main weakness? | Answer relevancy was much more variable, averaging approximately 0.48. This means the model was often faithful, but not always directly useful when retrieval was weak. |

The key finding is that the main weakness was not the LLM generation step. The main weakness was retrieval quality. When the retriever found strong evidence, the generated answer was useful and grounded. When retrieval returned weak or low similarity chunks, the LLM usually avoided hallucination but produced incomplete or non answer responses.

### 15.1.3 Failure Analysis

The system produced several clear failure cases during testing:

| Failure Case | What Happened | Why It Matters | Future Fix |
|---|---|---|---|
| Weak retrieval for broad strategy questions | Some questions about AI as a strategic investment area returned low similarity scores and no direct answer. | The answer becomes too general or incomplete when retrieved evidence does not directly match the question. | Use query expansion, hybrid retrieval, and more targeted AI strategy filters. |
| Company specific retrieval weakness | The SIFCO Industries question exposed a metadata limitation. The system does not deeply route company names, tickers, or CIK values during retrieval. | This limits usefulness for company-level financial analysis and SEC research. | Prepend company name, ticker, CIK, and section ID to every chunk before embedding. Add metadata filtering before vector search. |
| Low answer relevancy despite high grounding | Some answers were faithful because they refused to invent information, but they received low answer relevancy scores because they did not directly answer the prompt. | The system is safe but sometimes not useful enough. | Improve retrieval coverage, use reranking more aggressively, and increase top_k only when scores are strong. |
| Domain filtering may still be too broad | The AI corpus uses Item 1, Item 1A, and Item 7, but not every chunk from these sections is directly about AI. | Irrelevant chunks can dilute retrieval quality and reduce precision. | Build a stricter AI-only corpus using keyword filtering and manual validation before embedding. |
| Similarity only FAISS search has limitations | Dense retrieval may miss exact company names, tickers, or specialized financial phrases. | Important evidence can be missed even when it exists in the corpus. | Combine FAISS with BM25 or keyword search to create a hybrid retrieval system. |

### Evaluation Conclusion

The RAG system works best for broad AI related questions where SEC filings contain explicit language about risks, regulation, privacy, compliance, customer experience, or competitive threats. It performs less consistently for company specific questions and broad strategic investment questions. The model usually remains grounded, but retrieval quality determines whether the final answer is useful. The most important improvement should be stronger metadata aware retrieval, hybrid search, and tighter domain filtering before embedding.


In [ ]:
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# -------------------------------------------------------------------
# Load RAG output data if the variables are not already in memory.
# This allows the evaluation section to run independently after the
# earlier RAG query and RAGAS sections have saved their results.
# -------------------------------------------------------------------
try:
    df_qa_eval = df_qa_results.copy()
except NameError:
    possible_qa_paths = []
    if "OUTPUT_DIR" in globals():
        possible_qa_paths.append(OUTPUT_DIR / "domain_qa.csv")
    possible_qa_paths.append("domain_qa.csv")
    possible_qa_paths.append("/content/drive/MyDrive/SEC-10K-2024-ITEMS/domain_qa.csv")
    possible_qa_paths.append("/mnt/data/domain_qa.csv")

    df_qa_eval = None
    for p in possible_qa_paths:
        try:
            if Path(p).exists():
                df_qa_eval = pd.read_csv(p)
                break
        except Exception:
            pass
    if df_qa_eval is None:
        raise FileNotFoundError("Could not find domain_qa.csv. Run Step 9 first or check OUTPUT_DIR.")

try:
    df_ragas_eval = df_ragas_results.copy()
except NameError:
    possible_ragas_paths = []
    if "OUTPUT_DIR" in globals():
        possible_ragas_paths.append(OUTPUT_DIR / "df_ragas_results.csv")
    possible_ragas_paths.append("df_ragas_results.csv")
    possible_ragas_paths.append("/content/drive/MyDrive/SEC-10K-2024-ITEMS/df_ragas_results.csv")
    possible_ragas_paths.append("/mnt/data/df_ragas_results.csv")

    df_ragas_eval = None
    for p in possible_ragas_paths:
        try:
            if Path(p).exists():
                df_ragas_eval = pd.read_csv(p)
                break
        except Exception:
            pass
    if df_ragas_eval is None:
        raise FileNotFoundError("Could not find df_ragas_results.csv. Run Step 10 first or check OUTPUT_DIR.")

# -------------------------------------------------------------------
# Helper functions for parsing scores and classifying performance.
# -------------------------------------------------------------------
def parse_score_list(score_value):
    """Convert the saved Chunk_Scores string/list into a numeric list."""
    if isinstance(score_value, list):
        return [float(x) for x in score_value]
    if pd.isna(score_value):
        return []
    try:
        parsed = ast.literal_eval(str(score_value))
        return [float(x) for x in parsed]
    except Exception:
        return []

def retrieval_label(top_1_score, mean_top_5_score):
    """Manual retrieval relevance label based on observed similarity scores."""
    if top_1_score >= 0.70 and mean_top_5_score >= 0.35:
        return "Strong"
    elif top_1_score >= 0.35:
        return "Moderate / Partial"
    else:
        return "Weak"

def grounding_label(faithfulness):
    """Grounding label based on RAGAS faithfulness score."""
    if faithfulness >= 0.85:
        return "Well grounded"
    elif faithfulness >= 0.50:
        return "Partially grounded"
    else:
        return "Not adequately grounded"

def unsupported_claims_label(faithfulness):
    """Flag potential unsupported claims based on faithfulness."""
    if faithfulness >= 0.85:
        return "No obvious unsupported claims"
    elif faithfulness >= 0.50:
        return "Possible unsupported synthesis"
    else:
        return "High unsupported-claim risk"

# -------------------------------------------------------------------
# Create a reproducible manual evaluation table.
# -------------------------------------------------------------------
df_qa_eval["Score_List"] = df_qa_eval["Chunk_Scores"].apply(parse_score_list)
df_qa_eval["Top_1_Score"] = df_qa_eval["Score_List"].apply(lambda x: round(max(x), 3) if x else np.nan)
df_qa_eval["Mean_Top_5_Score"] = df_qa_eval["Score_List"].apply(lambda x: round(float(np.mean(x)), 3) if x else np.nan)

# Merge RAGAS scores onto the retrieval table.
ragas_cols = df_ragas_eval[["user_input", "faithfulness", "answer_relevancy"]].rename(columns={"user_input": "Question"})
manual_eval = df_qa_eval.merge(ragas_cols, on="Question", how="left")

manual_eval["Retrieval Relevance"] = manual_eval.apply(
    lambda row: retrieval_label(row["Top_1_Score"], row["Mean_Top_5_Score"]),
    axis=1
)
manual_eval["Captured Right Domain Evidence?"] = manual_eval["Retrieval Relevance"].map({
    "Strong": "Yes",
    "Moderate / Partial": "Partially",
    "Weak": "No / Limited"
})
manual_eval["Grounding Assessment"] = manual_eval["faithfulness"].apply(grounding_label)
manual_eval["Unsupported Claims Check"] = manual_eval["faithfulness"].apply(unsupported_claims_label)

section15_manual_eval = manual_eval[[
    "Question",
    "Top_1_Score",
    "Mean_Top_5_Score",
    "Retrieval Relevance",
    "Captured Right Domain Evidence?",
    "faithfulness",
    "answer_relevancy",
    "Grounding Assessment",
    "Unsupported Claims Check"
]].copy()

print("Section 15.1 — Manual Evaluation of Retrieval Relevance and Grounding")
display(section15_manual_eval)

# Save the table for submission evidence.
try:
    manual_eval_path = OUTPUT_DIR / "section15_manual_evaluation.csv"
except NameError:
    manual_eval_path = "section15_manual_evaluation.csv"
section15_manual_eval.to_csv(manual_eval_path, index=False)
print(f"Manual evaluation table saved to: {manual_eval_path}")

In [ ]:
from IPython.display import Markdown, display

# -------------------------------------------------------------------
# Generate an interpretation summary directly from the computed table.
# -------------------------------------------------------------------
faithfulness_mean = section15_manual_eval["faithfulness"].mean()
answer_relevancy_mean = section15_manual_eval["answer_relevancy"].mean()
strong_count = (section15_manual_eval["Retrieval Relevance"] == "Strong").sum()
moderate_count = (section15_manual_eval["Retrieval Relevance"] == "Moderate / Partial").sum()
weak_count = (section15_manual_eval["Retrieval Relevance"] == "Weak").sum()
well_grounded_count = (section15_manual_eval["Grounding Assessment"] == "Well grounded").sum()

summary_md = f"""
### 15.2 Grounding and Retrieval Summary

The manual evaluation table shows that retrieval quality is mixed across the test questions. Out of {len(section15_manual_eval)} questions, **{strong_count}** were classified as strong retrieval cases, **{moderate_count}** were moderate or partial retrieval cases, and **{weak_count}** were weak retrieval cases.

The average **faithfulness** score is **{faithfulness_mean:.2f}**, meaning the generated answers are usually grounded in the retrieved SEC 10-K excerpts. The average **answer relevancy** score is **{answer_relevancy_mean:.2f}**, which shows that some answers were faithful but not always fully responsive when retrieval was weak.

Overall, the main issue is not excessive hallucination. The main issue is that the retriever sometimes fails to provide sufficiently relevant context, causing the LLM to produce cautious, incomplete, or low-relevancy answers.
"""

display(Markdown(summary_md))

In [ ]:
# -------------------------------------------------------------------
# Generate at least three failure cases using rule-based checks from
# retrieval scores, RAGAS scores, and the question text.
# -------------------------------------------------------------------
failure_cases = []

# Failure 1: weak retrieval based on similarity scores.
weak_examples = section15_manual_eval[section15_manual_eval["Retrieval Relevance"] == "Weak"]
if not weak_examples.empty:
    q = weak_examples.iloc[0]["Question"]
    failure_cases.append({
        "Failure Case": "Weak retrieval",
        "Evidence from Evaluation": f"Example question: {q}",
        "What Happened": "The top retrieved chunks had low similarity scores, so the retrieved evidence was not strong enough.",
        "Why It Matters": "Weak retrieval leads to vague or incomplete generated answers even when the LLM follows the grounding instructions.",
        "Future Fix": "Use query expansion, hybrid dense/sparse search, or stricter AI-domain filtering before embedding."
    })

# Failure 2: low answer relevancy despite grounding.
low_relevancy_examples = section15_manual_eval[section15_manual_eval["answer_relevancy"] < 0.50]
if not low_relevancy_examples.empty:
    q = low_relevancy_examples.iloc[0]["Question"]
    failure_cases.append({
        "Failure Case": "Low answer relevancy",
        "Evidence from Evaluation": f"Example question: {q}",
        "What Happened": "The answer was often safe and grounded, but it did not fully answer the user's question.",
        "Why It Matters": "A RAG system can be faithful but still not useful if the retrieved context does not match the question closely enough.",
        "Future Fix": "Improve retrieval coverage, tune top_k by question type, and use reranking or keyword filtering."
    })

# Failure 3: company-specific retrieval weakness.
company_specific_examples = section15_manual_eval[
    section15_manual_eval["Question"].str.contains("SIFCO|company|Apple|Microsoft|ticker|CIK", case=False, na=False)
]
if not company_specific_examples.empty:
    q = company_specific_examples.iloc[0]["Question"]
    failure_cases.append({
        "Failure Case": "Company-specific retrieval weakness",
        "Evidence from Evaluation": f"Example question: {q}",
        "What Happened": "The system does not deeply route company names, tickers, or CIK metadata during retrieval.",
        "Why It Matters": "This limits the notebook's usefulness for company-level financial analysis.",
        "Future Fix": "Prepend company name, ticker, CIK, and section ID to each chunk before embedding and add metadata filtering."
    })

# Failure 4: weak grounding when faithfulness is low.
low_faithfulness_examples = section15_manual_eval[section15_manual_eval["faithfulness"] < 0.75]
if not low_faithfulness_examples.empty:
    q = low_faithfulness_examples.iloc[0]["Question"]
    failure_cases.append({
        "Failure Case": "Grounding weakness",
        "Evidence from Evaluation": f"Example question: {q}",
        "What Happened": "The faithfulness score was below the acceptable threshold, indicating that the answer was not fully supported by retrieved evidence.",
        "Why It Matters": "Grounding failures increase hallucination risk and reduce trust in the generated answer.",
        "Future Fix": "Use a stricter prompt, require source-level citations, and reject answers when evidence is insufficient."
    })

# Failure 5: broad corpus / domain noise.
if weak_count > 0 or moderate_count > 0:
    failure_cases.append({
        "Failure Case": "Domain filtering too broad",
        "Evidence from Evaluation": f"{weak_count} weak retrieval cases and {moderate_count} moderate/partial retrieval cases were observed.",
        "What Happened": "Some chunks from Items 1, 1A, and 7 were only indirectly related to AI strategy or AI risk.",
        "Why It Matters": "Broad domain filtering can add noise to the vector index and reduce retrieval precision.",
        "Future Fix": "Create a stricter AI-only filtered corpus using AI keywords, metadata filters, and manual validation."
    })

section15_failure_analysis = pd.DataFrame(failure_cases).drop_duplicates(subset=["Failure Case"]).head(5)

print("Section 15.3 — Failure Analysis")
display(section15_failure_analysis)

# Save failure analysis for submission evidence.
try:
    failure_path = OUTPUT_DIR / "section15_failure_analysis.csv"
except NameError:
    failure_path = "section15_failure_analysis.csv"
section15_failure_analysis.to_csv(failure_path, index=False)
print(f"Failure analysis table saved to: {failure_path}")

In [ ]:
# -------------------------------------------------------------------
# Final automatically generated Section 15 conclusion.
# -------------------------------------------------------------------
most_common_failure = section15_failure_analysis.iloc[0]["Failure Case"] if not section15_failure_analysis.empty else "No major failure case detected"

conclusion_md = f"""
### Section 15 Conclusion

This evaluation satisfies the required dimensions of the project: **retrieval relevance**, **grounding**, and **failure analysis**. The strongest result is grounding: the average faithfulness score is **{faithfulness_mean:.2f}**, and **{well_grounded_count}** out of {len(section15_manual_eval)} answers were classified as well grounded.

The biggest limitation is **{most_common_failure.lower()}**. In future work, the RAG system should improve retrieval through metadata injection, hybrid search, query expansion, and better domain filtering before scaling into the larger Project 3 system.
"""

display(Markdown(conclusion_md))

## Design Trade-offs

Limitation and Future Improvements: A key limitation of the current RAG system is that company metadata (such as the company name or ticker) is not deeply integrated into the retrieval process. Because the vector search relies purely on semantic similarity of the raw chunk text, the system struggles with company-specific queries (e.g., "What is Apple's AI strategy?"). Consequently, the questions must be structured as broad, corpus-wide inquiries rather than targeted searches, which restricts the system's utility for granular financial analysis and lowers Answer Relevancy scores when specific companies are queried but not successfully retrieved. In future iterations, this can be resolved by implementing Metadata Injection (prepending the company name and section explicitly into every text chunk before generating its embedding).